# Zephyr SOTA World-Model RL: Amortized Actor-Critic (Colab Runner)

This notebook trains and benchmarks the **Amortized Actor-Critic World Model** (Dreamer/Zephyr architecture) on Crafter from raw visual interaction (`64×64×3` RGB).

### Strict Non-Negotiable Protocol (AGENTS.md / GEMINI.md):
1. **Zero Hardcoded Recipes:** Zero lookup tables, dictionaries, or cheat rules (`AST grep count: 0 matches`).
2. **Pure Visual Inference Boundary:** The agent's `act()` function receives strictly `obs_uint8`. Zero privileged inventory or achievements enter the decision path.
3. **Zero Test-Time CEM Search:** Execution is an instant, closed-loop neural policy forward pass $\pi_\theta(a|s_t)$ ($<1\text{ms}$), completely eliminating test-time latent compounding drift.
4. **65-Bin Symlog Two-Hot Critic:** Propagates multi-step $\lambda$-returns over horizon $H=15$, learning the value of intermediate preparation (e.g. collecting 2 wood) even when immediate reward is $0.0$.
5. **Persistent Drive Storage:** All checkpoints and replay shards stream directly to `/content/drive/MyDrive/learn-a-world/zephyr/` to survive Colab disconnections.

## 1. Mount Google Drive and Setup Dedicated Zephyr Directory

In [ ]:
from google.colab import drive
from pathlib import Path
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Configure dedicated Zephyr model directory
MASTER_DRIVE_DIR = Path('/content/drive/MyDrive/learn-a-world')
ZEPHYR_DRIVE_DIR = MASTER_DRIVE_DIR / 'zephyr'

# Subdirectories for checkpoints, shards, and evaluation reports
for sub in ['checkpoints', 'runs/s1/ckpt', 'runs/s1/reports', 'runs/s1/data']:
    (ZEPHYR_DRIVE_DIR / sub).mkdir(parents=True, exist_ok=True)

print("Google Drive mounted successfully.")
print(f"Master Drive root: {MASTER_DRIVE_DIR}")
print(f"Zephyr dedicated folder: {ZEPHYR_DRIVE_DIR}")

## 2. Sync Repository from GitHub

In [ ]:
%cd /content
import os

# Clone or pull latest main branch
if not os.path.exists("learn-a-world"):
    !git clone https://github.com/sahiltolani30/learn-a-world.git /content/learn-a-world
else:
    %cd /content/learn-a-world
    !git fetch origin
    !git pull origin main

%cd /content/learn-a-world
!git log -1 --oneline

## 3. Install Dependencies & Hardware Audit

In [ ]:
# Install dependencies
!pip install --quiet crafter gymnasium pytest numpy torch torchvision

import sys
repo_root = '/content/learn-a-world'
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("WARNING: Running on CPU. For fast training, enable GPU via Runtime > Change runtime type > T4 / L4 / A100.")

## 4. Run Architectural & Anti-Cheating Invariants Test Suite

Runs the isolation and integrity verification tests:
- Zero imports from legacy heuristic modules
- Zero cheat recipe tables or hardcoded dictionaries
- Pinned 64×64 uint8 observation interface
- Two-hot categorical symlog critic encoding and loss
- Closed-loop latent imagination rollout gradient flow

In [ ]:
%cd /content/learn-a-world
!pytest tests/test_zephyr_*.py tests/e2e/tier4_real_world/ -v

## 5. Launch Joint Actor-Critic + World Model Training Loop (Stage 1)

Trains the joint system on GPU with automated Drive persistence:
- Vectors 4 parallel environments for high collection throughput
- Joint optimization: World Model + 65-bin Symlog Critic + Amortized Actor Policy
- Auto-resumes from `/content/drive/MyDrive/learn-a-world/zephyr/runs/s1/state.pt` if reconnected

In [ ]:
%cd /content/learn-a-world

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Launch Stage 1 Training
!python3 -m zephyr.colab.run_s1 \
    --drive-dir /content/drive/MyDrive/learn-a-world/zephyr \
    --total-frames 500000 \
    --screen-every 100000 \
    --frames-per-cycle 5000 \
    --train-steps-per-cycle 2500 \
    --num-envs 4 \
    --device $device

## 6. Run Official 50-Episode Locked Benchmark Screening

Runs the strict visual inference evaluation:
- 50 locked evaluation seeds (`2050..2099`)
- Strictly `(obs_uint8)` input path
- Evaluates using the official `compute_crafter_score` whitelist scorer
- Records full provenance (git commit, checkpoint SHA-256) into JSON report

In [ ]:
%cd /content/learn-a-world

report_out = "/content/drive/MyDrive/learn-a-world/zephyr/runs/s1/reports/official_benchmark_screen.json"
wm_ckpt = "/content/drive/MyDrive/learn-a-world/zephyr/runs/s1/ckpt/zephyr_wm_latest.pt"
heads_ckpt = "/content/drive/MyDrive/learn-a-world/zephyr/runs/s1/ckpt/zephyr_heads_latest.pt"

!python3 -c "
from zephyr.screen import run_screen
import os

wm_path = '$wm_ckpt'
heads_path = '$heads_ckpt'
out_path = '$report_out'

if os.path.exists(wm_path) and os.path.exists(heads_path):
    print('Running locked 50-episode screen on GPU/CPU...')
    report = run_screen(
        checkpoint_wm=wm_path,
        checkpoint_heads=heads_path,
        out_path=out_path,
        num_episodes=50,
        seed_start=2050,
        max_steps=1000,
        agent_mode='actor',
        device='cuda' if torch.cuda.is_available() else 'cpu'
    )
    print(f'Benchmark Complete! Official Crafter Score: {report["crafter_score"]:.2f}%')
    print(f'Mean Survival Length: {report["mean_survival_length"]:.1f} steps')
    print(f'Full Report saved to: {out_path}')
else:
    print('Checkpoints not found. Ensure training has completed at least one cycle.')
"